In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from jppype import vscode_theme

from fundus_vessels_toolkit.segment_to_graph.models.dataset import VBranchDigraphDataset
from fundus_vessels_toolkit.segment_to_graph.models.losses import VBranchDigraphMiner
from fundus_vessels_toolkit.segment_to_graph.models.model import BranchDigraphModel
from fundus_vessels_toolkit.segment_to_graph.models.trainer import DigraphGNNTrainer
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from fundus_vessels_toolkit.utils.tree import tree_connected_components

vscode_theme()


HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [3]:
import datetime

PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV", "LES-AV", "INSPIRE"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]
opts = dict(resize_to=1024, root="tmp/DATA2", ignore_recent=datetime.datetime(2026, 3, 8))
dataset = VBranchDigraphDataset.load_from_dirs(RAW, TOPO, av_dir=AV, **opts)
train_set, val_set, test_set = dataset.split_loaders(train_ratio=0.7, val_ratio=0.15)

Found 291 branch digraphs...


Processing...
Done!
Preloading dataset: 100%|██████████| 291/291 [00:27<00:00, 10.58it/s]


## Visualize result from pred table


In [4]:
def parse_arborescence(preds, idx, opti=False):
    pred = preds.loc[idx] if isinstance(idx, str) else preds.iloc[idx]
    b_parent = np.fromstring(pred[("opti_" if opti else "") + "parent"], sep=",", dtype=int)
    b_dir = np.fromstring(pred[("opti_" if opti else "") + "dir"], sep=",", dtype=int)
    b_av = np.fromstring(pred["av"], sep=",", dtype=int)
    return pred.name, b_parent, b_dir, b_av


### Load model from checkpoint


In [5]:
model = (
    DigraphGNNTrainer.load_from_checkpoint("../../lightning_logs/cfelnidm/checkpoints/epoch=239-step=8160.ckpt")
    .model.cuda()
    .eval()
)

In [ ]:
ID = 45
with torch.inference_mode():
    out: BranchDigraphModel.Output = model(test_set.get(ID).cuda())
pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0
pred_parent, pred_dir, pred_av = out.optimal_tree

gt_digraph, _, od_yx, _ = test_set.get_sample(ID)
assert VBranchDigraph.has_fp_av_p(gt_digraph)
valid_branch = ~gt_digraph.branch_fp()
av_gt = gt_digraph.branch_av_class() <= 1
print(((out.av_logit.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())
print(((pred_av.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())


/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:409: operator(): block: [6,0,0], thread: [43,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:409: operator(): block: [4,0,0], thread: [35,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:409: operator(): block: [6,0,0], thread: [109,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:409: operator(): block: [4,0,0], thread: [4,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:409: operator(): block: [7,0,0], thread: [113,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherK

AcceleratorError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
from pytorch_metric_learning import losses, distances


pairs = VBranchDigraphMiner(triplet=False)(out)
triplet = VBranchDigraphMiner(triplet=True)(out)

CosSim = distances.CosineSimilarity()
(
    losses.ContrastiveLoss(distance=CosSim, pos_margin=1, neg_margin=0)(out.b1_embedding, indices_tuple=pairs),
    losses.TripletMarginLoss()(out.b1_embedding, indices_tuple=triplet),
)

(tensor(0.9862, device='cuda:0'), tensor(0.0968, device='cuda:0'))

In [ ]:
m, pred_tree = test_set.show_tree_diff(
    out.name,
    pred_parent.numpy(force=True),
    pred_dir.numpy(force=True),
    out.fp_logit.numpy(force=True) > 0,
    pred_av.numpy(force=True) > 0,
)
m

71 97 0.5002683401107788
-1 97 1.999359905719757


GridBox(children=(HTML(value='<h3 style="text-align: center;">GT Tree: image13</h3>'), HTML(value='<h3 style="…

In [ ]:
m.views[0].goto(pred_tree.branch(177).midpoint().xy, scale=3)

In [ ]:
out.av_logit[211]

tensor(-5.6672, device='cuda:0')

In [ ]:
B_ID = 178
pred_av[B_ID], out.av_logit[B_ID], pred_parent[B_ID], out.max_parent()[B_ID]

(tensor(-5.0150, device='cuda:0'),
 tensor(-2.5944, device='cuda:0'),
 tensor(115, device='cuda:0'),
 tensor(115, device='cuda:0'))

In [ ]:
np.set_printoptions(linewidth=200, precision=2, suppress=True)
d = out.to_digraph()
d.lines_info(b1=97, sort_by_p=True).round(3).head(20)

/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:409: operator(): block: [4,0,0], thread: [35,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:409: operator(): block: [6,0,0], thread: [43,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:409: operator(): block: [4,0,0], thread: [4,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:409: operator(): block: [6,0,0], thread: [109,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:409: operator(): block: [4,0,0], thread: [72,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKe

AcceleratorError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import tqdm

from fundus_toolkits import AVLabel
from fundus_toolkits.utils.geometric import Point
from fundus_vessels_toolkit.segment_to_graph.av_tree_parsing import naive_infer_arborescence

name = []
pred_parent_acc = []
pred_dir_acc = []
pred_av_acc = []
opti_parent_acc = []
opti_dir_acc = []
opti_av_acc = []
baseline_parent_acc = []
baseline_dir_acc = []
baseline_av_acc = []
node_ratio = []
branch_ratio = []

eval_set = test_set

with torch.inference_mode():
    for i in tqdm.tqdm(range(len(eval_set))):
        gt_digraph, _, od_yx, _ = eval_set.get_sample(i)
        assert gt_digraph.branch_dir_p is not None
        assert gt_digraph.graph is not None, "Graph must be loaded to infer tree"

        node_ratio += [gt_digraph.graph.node_count / eval_set.graphs[i].node_count]
        branch_ratio += [gt_digraph.graph.branch_count / eval_set.graphs[i].branch_count]

        valid_branch = ~gt_digraph.branch_fp()
        av_gt = (gt_digraph.branch_av_class() <= 1)[valid_branch]
        od = Point.parse(od_yx)

        art_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.ART
        vei_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.VEI
        parent_base = -np.ones(gt_digraph.graph.branch_count, dtype=np.int_)
        dir_base = np.zeros(gt_digraph.graph.branch_count, dtype=np.bool_)
        parent_base[art_branch], dir_base[art_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=art_branch
        )
        parent_base[vei_branch], dir_base[vei_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=vei_branch
        )
        parent_base, dir_base = parent_base[valid_branch], dir_base[valid_branch]
        baseline_parent_acc.append((parent_base == gt_digraph.max_parent()[valid_branch]).mean())
        baseline_dir_acc.append((dir_base == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        baseline_av_acc.append((art_branch[valid_branch] == av_gt).mean())

        out = model(eval_set.get(i).cuda())
        name += [out.name]
        pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0

        pred_parent, pred_dir = pred_parent.numpy(force=True), pred_dir.numpy(force=True)
        pred_parent, pred_dir = pred_parent[valid_branch], pred_dir[valid_branch]
        av_logit = out.av_logit.numpy(force=True)[valid_branch]
        pred_parent_acc.append((pred_parent == gt_digraph.max_parent()[valid_branch]).mean())
        pred_dir_acc.append((pred_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        pred_av_acc.append(((av_logit > 0) == av_gt).mean())

        opti_parent, opti_dir, opti_av = out.optimal_tree
        opti_parent, opti_dir = opti_parent.numpy(force=True), opti_dir.numpy(force=True)
        opti_parent, opti_dir = opti_parent[valid_branch], opti_dir[valid_branch]
        opti_parent_acc.append((opti_parent == gt_digraph.max_parent()[valid_branch]).mean())
        opti_dir_acc.append((opti_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())

        opti_av = opti_av.numpy(force=True)[valid_branch]
        opti_av_acc.append(((opti_av > 0) == av_gt).mean())

  0%|          | 0/46 [00:00<?, ?it/s]

100%|██████████| 46/46 [00:14<00:00,  3.12it/s]


In [ ]:
np.mean(node_ratio), np.mean(branch_ratio)

(np.float64(1.4904167722713746), np.float64(1.4984898087851408))

In [ ]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

(np.float64(0.9178801846550065),
 np.float64(0.9232224978208828),
 np.float64(0.854172589406778))

In [ ]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

(np.float64(0.988149197560497),
 np.float64(0.9901218398462059),
 np.float64(0.9492600171861724))

In [ ]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

(np.float64(0.9494334131536296),
 np.float64(0.943272418401274),
 np.float64(0.9532570626720513))

In [ ]:
np.array(opti_av_acc)[23], np.array(pred_av_acc)[23]

(np.float64(0.9781931464174455), np.float64(0.9781931464174455))

In [ ]:
(np.array(pred_parent_acc) - np.array(pred_parent_acc)).argsort()[::-1]

array([45, 44, 43, 42, 41, 40, 39, 38, 37, 36, 35, 34, 33, 32, 31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10,  9,  8,  7,  6,  5,  4,  3,  2,  1,  0])

In [ ]:
name

['g_039',
 'g_016',
 'g_011',
 'g_050',
 'g_031',
 'g_007',
 'g_043',
 'g_003',
 '20060412_58497_0200_PP',
 '20051205_59351_0400_PP',
 '20051205_59538_0400_PP',
 '20060412_58471_0200_PP',
 '20060530_53062_0100_PP',
 '20060523_45235_0100_PP',
 '20060523_48199_0100_PP',
 '20051202_51488_0400_PP',
 '20060412_51952_0200_PP',
 '20051205_35305_0400_PP',
 '20060523_49681_0100_PP',
 '20060412_61251_0200_PP',
 '20060410_41767_0200_PP',
 '20060411_61478_0200_PP',
 '20051208_39243_0400_PP',
 '20051212_36548_0400_PP',
 '009_D',
 '018_N',
 '047_A',
 '032_A',
 '007_N',
 '030_A',
 '045_A',
 '043_N',
 '003_N',
 '016_N',
 '094_N',
 '017_G',
 '093_N',
 '087_D',
 '077_G',
 '42',
 '224',
 '12',
 '37',
 'image24',
 'image5',
 'image13']